In [27]:
from datetime import datetime, timezone

import requests
import wikipedia
from dotenv import find_dotenv, load_dotenv
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

from langchain.agents import AgentExecutor, tool
from langchain.agents.format_scratchpad import format_to_tool_messages
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.agent import AgentFinish
from langchain.schema.runnable import RunnablePassthrough

In [2]:
load_dotenv(find_dotenv("../../creds/.env"), verbose=True);

In [3]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="mistral-nemo:12b",
    temperature=0,
    verbose=True,
    extract_reasoning=True,
)

In [4]:
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")


@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> str:
    """Fetch current temperature for given coordinates."""

    BASE_URL = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1,
        "timezone": "GMT",
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.now(timezone.utc)
    time_list = [
        datetime.fromisoformat(time_str).replace(tzinfo=timezone.utc) for time_str in results["hourly"]["time"]
    ]
    temperature_list = results["hourly"]["temperature_2m"]

    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]

    return f"The current temperature is {current_temperature}°C"


get_current_temperature.invoke({"latitude": 13, "longitude": 14})

'The current temperature is 32.0°C'

In [5]:
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries"""

    page_titles = wikipedia.search(query)
    summaries = []

    for page_title in page_titles[:3]:
        try:
            wiki_page = wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            wikipedia.exceptions.PageError,
            wikipedia.exceptions.DisambiguationError,
        ):
            pass

    if not summaries:
        return "No good Wikipedia Search Result was found"

    return "\n\n".join(summaries)


print(search_wikipedia.invoke({"query": "langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Retrieval-augmented generation
Summary: Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information. With RAG, LLMs do not respond to user queries until they refer to a specified set of documents. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this helps LLM-based chatbots access internal company data or generate responses based on authoritative sources.
RAG improves large language models (LLMs) by incor

In [6]:
functions = [
    get_current_temperature,
    search_wikipedia,
]
tools = [
    convert_to_openai_function(get_current_temperature),
    convert_to_openai_function(search_wikipedia),
]
model = llm.bind_tools(tools=tools)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are helpful but sassy assistant"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)
chain = prompt | model | OpenAIToolsAgentOutputParser()

In [7]:
result_1 = chain.invoke(
    {
        "input": "what is the weather in Kyiv?",
        "agent_scratchpad": [],
    }
)

In [8]:
result_1[0]

ToolAgentAction(tool='get_current_temperature', tool_input={'latitude': 50.45, 'longitude': 30.52}, log="\nInvoking: `get_current_temperature` with `{'latitude': 50.45, 'longitude': 30.52}`\n\n\n", message_log=[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'mistral-nemo:12b', 'created_at': '2025-07-07T18:06:34.187458015Z', 'done': True, 'done_reason': 'stop', 'total_duration': 30864789035, 'load_duration': 29145945559, 'prompt_eval_count': 148, 'prompt_eval_duration': 551164311, 'eval_count': 36, 'eval_duration': 1161869617, 'model_name': 'mistral-nemo:12b'}, id='run--2d70d4a9-a0a0-48b1-b3c4-4985e570645c-0', tool_calls=[{'name': 'get_current_temperature', 'args': {'latitude': 50.45, 'longitude': 30.52}, 'id': '2411bce2-9c22-4f22-8e59-c89a1a6f6091', 'type': 'tool_call'}], usage_metadata={'input_tokens': 148, 'output_tokens': 36, 'total_tokens': 184})], tool_call_id='2411bce2-9c22-4f22-8e59-c89a1a6f6091')

In [9]:
result_1[0].message_log

[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'mistral-nemo:12b', 'created_at': '2025-07-07T18:06:34.187458015Z', 'done': True, 'done_reason': 'stop', 'total_duration': 30864789035, 'load_duration': 29145945559, 'prompt_eval_count': 148, 'prompt_eval_duration': 551164311, 'eval_count': 36, 'eval_duration': 1161869617, 'model_name': 'mistral-nemo:12b'}, id='run--2d70d4a9-a0a0-48b1-b3c4-4985e570645c-0', tool_calls=[{'name': 'get_current_temperature', 'args': {'latitude': 50.45, 'longitude': 30.52}, 'id': '2411bce2-9c22-4f22-8e59-c89a1a6f6091', 'type': 'tool_call'}], usage_metadata={'input_tokens': 148, 'output_tokens': 36, 'total_tokens': 184})]

In [10]:
observation = get_current_temperature.invoke(result_1[0].tool_input)
observation

'The current temperature is 29.8°C'

In [11]:
format_to_tool_messages([(result_1[0], observation)])

[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'mistral-nemo:12b', 'created_at': '2025-07-07T18:06:34.187458015Z', 'done': True, 'done_reason': 'stop', 'total_duration': 30864789035, 'load_duration': 29145945559, 'prompt_eval_count': 148, 'prompt_eval_duration': 551164311, 'eval_count': 36, 'eval_duration': 1161869617, 'model_name': 'mistral-nemo:12b'}, id='run--2d70d4a9-a0a0-48b1-b3c4-4985e570645c-0', tool_calls=[{'name': 'get_current_temperature', 'args': {'latitude': 50.45, 'longitude': 30.52}, 'id': '2411bce2-9c22-4f22-8e59-c89a1a6f6091', 'type': 'tool_call'}], usage_metadata={'input_tokens': 148, 'output_tokens': 36, 'total_tokens': 184}),
 ToolMessage(content='The current temperature is 29.8°C', tool_call_id='2411bce2-9c22-4f22-8e59-c89a1a6f6091')]

In [12]:
result_2 = chain.invoke(
    {
        "input": "what is the weather in Kyiv?",
        "agent_scratchpad": format_to_tool_messages([(result_1[0], observation)]),
    }
)
result_2

AgentFinish(return_values={'output': 'The current temperature in Kyiv is 29.8°C.'}, log='The current temperature in Kyiv is 29.8°C.')

In [14]:
agent_chain = (
    RunnablePassthrough.assign(agent_scratchpad=lambda x: format_to_tool_messages(x["intermediate_steps"])) | chain
)

In [15]:
def run_agent(user_input):
    tools = {
        "search_wikipedia": search_wikipedia,
        "get_current_temperature": get_current_temperature,
    }
    intermediate_steps = []

    while True:
        result = agent_chain.invoke(
            {
                "input": user_input,
                "intermediate_steps": intermediate_steps,
            }
        )

        if isinstance(result, AgentFinish):
            return result

        tool = tools[result[0].tool]
        observation = tool.run(result[0].tool_input)
        intermediate_steps.append((result[0], observation))

In [16]:
run_agent("What is the weather in Kyiv?")

AgentFinish(return_values={'output': 'The current temperature in Kyiv is 29.8°C.'}, log='The current temperature in Kyiv is 29.8°C.')

In [17]:
run_agent("What is langchain?")

AgentFinish(return_values={'output': 'LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. Its use-cases include document analysis and summarization, chatbots, and code analysis.\n\nAdditionally, LangChain has adopted the Model Context Protocol (MCP), an open standard introduced by Anthropic in November 2024 to standardize data sharing between AI systems and external tools. This allows for better integration with other systems and data sources.\n\nLangChain also uses Milvus, a distributed vector database developed by Zilliz, which is available as both open-source software and a cloud service. Milvus helps manage and search high-dimensional vector data efficiently.'}, log='LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. Its use-cases include document analysis and summarization, chatbots, and code analysis.\n\nAdditionally, LangChain has adopted

In [18]:
run_agent("hi!")

AgentFinish(return_values={'output': " Hi there! How can I assist you today? Let's keep it friendly, okay? 😊"}, log=" Hi there! How can I assist you today? Let's keep it friendly, okay? 😊")

In [19]:
agent_executor = AgentExecutor(agent=agent_chain, tools=functions, verbose=True)

In [20]:
agent_executor.invoke({"input": "what is langchain?"})



> Entering new AgentExecutor chain...

Invoking: `search_wikipedia` with `{'query': 'LangChain'}`


Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Model Context Protocol
Summary: The Model Context Protocol (MCP) is an open standard, open-source framework introduced by Anthropic in November 2024 to standardize the way artificial intelligence (AI) systems like large language models (LLMs) integrate and share data with external tools, systems, and data sources. MCP provides a universal interface for reading files, executing functions, and handling contextual prompts. Following its announcement, the protocol was adopted by major AI providers, including OpenAI and Google DeepMind.

Pag

{'input': 'what is langchain?',
 'output': 'LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. Its use-cases include document analysis and summarization, chatbots, and code analysis.\n\nAdditionally, LangChain supports the Model Context Protocol (MCP), an open standard introduced by Anthropic to standardize data sharing between AI systems and external tools. This allows for better integration of large language models with other software components.\n\nLangChain also leverages Milvus, a distributed vector database developed by Zilliz, which is used for efficient similarity search in high-dimensional data spaces.'}

In [21]:
agent_executor.invoke({"input": "what is the temperature in Kyiv??"})



> Entering new AgentExecutor chain...

Invoking: `get_current_temperature` with `{'latitude': 50.45, 'longitude': 30.52}`


The current temperature is 29.8°CThe current temperature in Kyiv is 29.8°C.

> Finished chain.


{'input': 'what is the temperature in Kyiv??',
 'output': 'The current temperature in Kyiv is 29.8°C.'}

In [22]:
agent_executor.invoke({"input": "My name is John"})



> Entering new AgentExecutor chain...
Hello John! How can I assist you today? Let's make this conversation great.

> Finished chain.


{'input': 'My name is John',
 'output': "Hello John! How can I assist you today? Let's make this conversation great."}

In [23]:
agent_executor.invoke({"input": "What is my name?"})



> Entering new AgentExecutor chain...
Your name is Assistant. How can I assist you today, Assistant?

> Finished chain.


{'input': 'What is my name?',
 'output': 'Your name is Assistant. How can I assist you today, Assistant?'}

In [58]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are helpful but sassy assistant"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)
prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')]

In [59]:
agent_chain = (
    RunnablePassthrough.assign(agent_scratchpad=lambda x: format_to_tool_messages(x["intermediate_steps"]))
    | prompt
    | model
    | OpenAIToolsAgentOutputParser()
)

In [60]:
memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

In [61]:
agent_executor = AgentExecutor(agent=agent_chain, tools=functions, memory=memory, verbose=True)
agent_executor

AgentExecutor(memory=ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[]), return_messages=True, memory_key='chat_history'), verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMess

In [62]:
agent_executor.invoke({"input": "my name is bob"})



> Entering new AgentExecutor chain...
Hello Bob! How can I assist you today? Let's make your day great.

> Finished chain.


{'input': 'my name is bob',
 'chat_history': [HumanMessage(content='my name is bob', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello Bob! How can I assist you today? Let's make your day great.", additional_kwargs={}, response_metadata={})],
 'output': "Hello Bob! How can I assist you today? Let's make your day great."}

In [63]:
agent_executor.invoke({"input": "I live in Kyiv"})



> Entering new AgentExecutor chain...
 Got it! Do you need help with anything specific, Bob?

> Finished chain.


{'input': 'I live in Kyiv',
 'chat_history': [HumanMessage(content='my name is bob', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello Bob! How can I assist you today? Let's make your day great.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='I live in Kyiv', additional_kwargs={}, response_metadata={}),
  AIMessage(content=' Got it! Do you need help with anything specific, Bob?', additional_kwargs={}, response_metadata={})],
 'output': ' Got it! Do you need help with anything specific, Bob?'}

In [64]:
agent_executor.invoke({"input": "Who was the very first recorded namesake of mine?"})



> Entering new AgentExecutor chain...

Invoking: `search_wikipedia` with `{'query': 'Bob (given name)'}`


Page: Bob (given name)
Summary: Bob is a male given name or a hypocorism, usually of Robert; and sometimes a diminutive of Bobby.
The name most likely originated from the hypocorism Rob, short for Robert. Rhyming names were popular in the Middle Ages, so Richard became Rick, Hick, or Dick, William became Will, Gill, or Bill, and Robert became Rob, Hob, Dob, Nob, or Bob.
The name Bob has shrunken in popularity in the past 60 years. In 1960, nearly 3,000 babies in the United States were given the name Bob, compared to fewer than 50 in 2000.

Page: Given name
Summary: A given name (also known as a forename or first name) is the part of a personal name that identifies a person, potentially with a middle name as well, and differentiates that person from the other members of a group (typically a family or clan) who have a common surname. The term given name refers to a name usually be

{'input': 'Who was the very first recorded namesake of mine?',
 'chat_history': [HumanMessage(content='my name is bob', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello Bob! How can I assist you today? Let's make your day great.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='I live in Kyiv', additional_kwargs={}, response_metadata={}),
  AIMessage(content=' Got it! Do you need help with anything specific, Bob?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Who was the very first recorded namesake of mine?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Based on the information provided by Wikipedia, the very first recorded namesake of yours, Bob, was Pabi, Prince of Lachish in central Israel, who lived around 1350 BC. He sent a letter to Pharaoh Akhenaton mentioning Rapha-el, which is considered one of the earliest known instances of the name Bob or its variant forms.', additional_kwargs={}, resp

In [65]:
agent_executor.invoke({"input": "What is the temperature in my town of residence?"})



> Entering new AgentExecutor chain...

Invoking: `get_current_temperature` with `{'latitude': 50.4501, 'longitude': 30.5234}`


The current temperature is 28.3°CThe current temperature in Kyiv is 28.3°C, Bob. Is there anything else you'd like to know about the weather?

> Finished chain.


{'input': 'What is the temperature in my town of residence?',
 'chat_history': [HumanMessage(content='my name is bob', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hello Bob! How can I assist you today? Let's make your day great.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='I live in Kyiv', additional_kwargs={}, response_metadata={}),
  AIMessage(content=' Got it! Do you need help with anything specific, Bob?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Who was the very first recorded namesake of mine?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Based on the information provided by Wikipedia, the very first recorded namesake of yours, Bob, was Pabi, Prince of Lachish in central Israel, who lived around 1350 BC. He sent a letter to Pharaoh Akhenaton mentioning Rapha-el, which is considered one of the earliest known instances of the name Bob or its variant forms.', additional_kwargs={}, respo